# ** Book Recommendation System**

A book recommendation system is a type of recommendation system where we have to recommend similar books to the reader based on his interest. The books recommendation system is used by online websites which provide ebooks like google play books, open library, good Read’s, etc.

In [1]:
# To get a list of all available files in the specified directory
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/book-recommender-system/BX-Books.csv
/kaggle/input/book-recommender-system/BX-Book-Ratings.csv
/kaggle/input/book-recommender-system/BX-Users.csv


In [43]:
# importing necessary libraries!

import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.sparse import csr_matrix
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from surprise import SVD, Dataset, Reader
from surprise.model_selection import cross_validate

In [3]:
#importing the csv files!!

warnings.filterwarnings("ignore")
books = pd.read_csv("/kaggle/input/book-recommender-system/BX-Books.csv", sep=';', encoding="latin-1", on_bad_lines='skip')
users = pd.read_csv("/kaggle/input/book-recommender-system/BX-Users.csv", sep=';', encoding="latin-1", on_bad_lines='skip')
ratings = pd.read_csv("/kaggle/input/book-recommender-system/BX-Book-Ratings.csv", sep=';', encoding="latin-1", on_bad_lines='skip')


**Preprocessing Data**

Now in the books file, we have some extra columns which are not required for our task like image URLs. And we will rename the columns of each file as the name of the column contains space, and uppercase letters so we will correct as to make it easy to use.


In [4]:
books = books[['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication', 'Publisher']]
books.rename(columns = {'Book-Title':'title', 'Book-Author':'author', 'Year-Of-Publication':'year', 'Publisher':'publisher'}, inplace=True)
users.rename(columns = {'User-ID':'user_id', 'Location':'location', 'Age':'age'}, inplace=True)
ratings.rename(columns = {'User-ID':'user_id', 'Book-Rating':'rating'}, inplace=True)

**Problem Statement:**
Build a book recommendation system that uses collaborative filtering to recommend books to users based on shared reading patterns and preferences.

**In this system:**

* If a user has shown interest in specific books that overlap with books enjoyed by other users, we will identify these shared interests.

* When a user (say, User A) has read and enjoyed books X and Y, and another user (User B) has also read and liked X and Y but hasn’t read Z (a book liked by User A), the system should recommend book Z to User B.

This approach will leverage collaborative filtering to suggest books that a user hasn't yet explored but might enjoy, based on the behavior and preferences of similar users.


**Expected Outcome:**
To develop a recommendation system that effectively suggests books by identifying and leveraging latent similarities in users’ preferences and reading histories.

**Expected Outcome:**
A system that provides personalized book recommendations, helping users discover new books that align with their preferences based on shared interests with other readers.

# Data Preprocessing

Filter Ratings Data:
* Remove rows with missing or invalid data (like null ISBNs, user_ids, or ratings).
* Filter out users or books with very few ratings to focus on more active users and popular books.

In [5]:
ratings.max()

user_id       278854
ISBN       Ô½crosoft
rating            10
dtype: object

In [6]:
'''
1. Filter the ratings DataFrame for active users and popular books.
2. Merge the filtered ratings with book data to include details like title and author.
3. Count ratings per title to focus on books with a significant number of ratings (e.g., 50 or more).
4. Remove duplicates so each (user_id, title) pair appears only once.
'''

# Step 1: Filter Users with 200+ Ratings
user_ratings_count = ratings['user_id'].value_counts()
ratings = ratings[ratings['user_id'].isin(user_ratings_count[user_ratings_count >= 200].index)]

# Step 2: Filter Books with 10+ Ratings
book_ratings_count = ratings['ISBN'].value_counts()
ratings = ratings[ratings['ISBN'].isin(book_ratings_count[book_ratings_count >= 10].index)]

# Step 3: Merge Ratings with Book Information
# This assumes 'ratings' has columns 'ISBN' and 'user_id' and 'books' has 'ISBN' along with other book details.
rating_with_books = ratings.merge(books, on='ISBN')

# Step 4: Count Ratings per Book Title
number_rating = rating_with_books.groupby('title')['rating'].count().reset_index()
number_rating.rename(columns={'rating': 'number_of_ratings'}, inplace=True)

# Step 5: Merge the Counts into the Ratings Data
final_rating = rating_with_books.merge(number_rating, on='title')

# Step 6: Filter Books with 50+ Ratings
final_rating = final_rating[final_rating['number_of_ratings'] >= 50]

# Step 7: Remove Duplicate User-Title Ratings
final_rating.drop_duplicates(['user_id', 'title'], inplace=True)

# Output the final DataFrame to verify
print("Final rating shape after all filtering:", final_rating.shape)
print("Sample of final rating data:\n", final_rating.head())


Final rating shape after all filtering: (53927, 8)
Sample of final rating data:
     user_id        ISBN  rating  \
0    277427  002542730X      10   
6    277427  0060930535       0   
8    277427  0060934417       0   
9    277427  0061009059       9   
14   277427  0140067477       0   

                                                title              author  \
0   Politically Correct Bedtime Stories: Modern Ta...   James Finn Garner   
6                       The Poisonwood Bible: A Novel  Barbara Kingsolver   
8                                  Bel Canto: A Novel        Ann Patchett   
9   One for the Money (Stephanie Plum Novels (Pape...     Janet Evanovich   
14                                    The Tao of Pooh       Benjamin Hoff   

    year                  publisher  number_of_ratings  
0   1994  John Wiley &amp; Sons Inc                 80  
6   1999                  Perennial                133  
8   2002                  Perennial                108  
9   1995         

In [30]:
final_rating

,user_id,ISBN,rating,title,author,year,publisher,number_of_ratings
0,277427,002542730X,10,Politically Correct Bedtime Stories: Modern Ta...,James Finn Garner,1994,John Wiley &amp; Sons Inc,80
6,277427,0060930535,0,The Poisonwood Bible: A Novel,Barbara Kingsolver,1999,Perennial,133
8,277427,0060934417,0,Bel Canto: A Novel,Ann Patchett,2002,Perennial,108
9,277427,0061009059,9,One for the Money (Stephanie Plum Novels (Pape...,Janet Evanovich,1995,HarperTorch,108
14,277427,0140067477,0,The Tao of Pooh,Benjamin Hoff,1983,Penguin Books,68
...,...,...,...,...,...,...,...,...
169690,275970,0860074382,10,84 Charing Cross Road,Helene Hanff,0,Warner Books> C/o Little Br,51
169697,275970,140003065X,0,A Fine Balance,Rohinton Mistry,2001,Vintage Books USA,55
169699,275970,1400031354,0,Tears of the Giraffe (No.1 Ladies Detective Ag...,Alexander McCall Smith,2002,Anchor,84
169700,275970,1400031362,0,Morality for Beautiful Girls (No.1 Ladies Dete...,Alexander McCall Smith,2002,Anchor,60


# Modeling and Predictions

In [31]:
final_rating.info()

<class 'pandas.core.frame.DataFrame'>
Index: 53927 entries, 0 to 169709
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   user_id            53927 non-null  int64 
 1   ISBN               53927 non-null  object
 2   rating             53927 non-null  int64 
 3   title              53927 non-null  object
 4   author             53927 non-null  object
 5   year               53927 non-null  object
 6   publisher          53927 non-null  object
 7   number_of_ratings  53927 non-null  int64 
dtypes: int64(3), object(5)
memory usage: 3.7+ MB


**Create a Pivot Table**

In [32]:
book_pivot = final_rating.pivot_table(columns='user_id', index='title', values="rating")
book_pivot.fillna(0, inplace=True)

In [33]:
book_pivot.head()

user_id,254,2276,2766,2977,3363,4017,4385,6242,6251,6323,...,274004,274061,274301,274308,274808,275970,277427,277478,277639,278418
title,,,,,,,,,,,,,,,,,,,,,
1984,9.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1st to Die: A Novel,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2nd Chance,0.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4 Blondes,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
84 Charing Cross Road,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,10.0,0.0,0.0,0.0,0.0


**Convert Pivot Table to Sparse Matrix**
* Using a sparse matrix is efficient as it reduces memory usage by ignoring zero entries. This is essential given the number of zeros in the dataset.

In [34]:
book_sparse = csr_matrix(book_pivot)

**Train the Nearest Neighbors Model**
Using **NearestNeighbors** with the brute algorithm and the cosine metric, which generally performs well for recommendation systems.

In [35]:
# Initialize the NearestNeighbors model
model = NearestNeighbors(algorithm='brute', metric='cosine')
model.fit(book_sparse)

NearestNeighbors(algorithm='brute', metric='cosine')

**Make Predictions**

* Now we’ll make a prediction for a specific book (e.g., Harry Potter). To do this:

* Pass in the book’s index to the model’s kneighbors function. Retrieve the closest matches.

In [36]:
# Find the index of the specific book in the pivot table
book_index = 333  # Example index for "Harry Potter"

# Get the top 5 closest books
distances, suggestions = model.kneighbors(book_pivot.iloc[book_index, :].values.reshape(1, -1), n_neighbors=5)


**Print the Suggested Books**

We can now print the titles of the suggested books by using the indices returned in suggestions.

In [37]:
print("Recommended books based on the input book:")
for i in range(len(suggestions[0])):
    print(book_pivot.index[suggestions[0][i]])

Recommended books based on the input book:
One Door Away from Heaven
By the Light of the Moon
Fear Nothing
Everything's Eventual : 14 Dark Tales
Pet Sematary


# Trainng and Testing

**Create a Train/Test Split**
* For each user, we’ll split their ratings into training and testing sets. We’ll use 80% of each user's ratings for training and the remaining 20% for testing.

In [38]:
def train_test_split_ratings(data, test_size=0.2, min_ratings=2, random_state=42):
    """
    Splits ratings data into training and testing sets for each user.
    
    Parameters:
    - data (pd.DataFrame): The full ratings dataset with columns 'user_id' and 'rating'.
    - test_size (float): Proportion of each user's ratings to include in the test set.
    - min_ratings (int): Minimum number of ratings a user must have to be included in the split.
    - random_state (int): Random state for reproducibility of the split.

    Returns:
    - train_data (pd.DataFrame): The training set with a subset of ratings for each user.
    - test_data (pd.DataFrame): The testing set with the remaining ratings for each user.
    """
    train_data = []
    test_data = []
    skipped_users = 0

    # Split each user's ratings if they meet the minimum number of ratings
    for user_id, user_ratings in data.groupby('user_id'):
        if len(user_ratings) < min_ratings:
            skipped_users += 1
            continue  # Skip users with fewer than the minimum ratings

        # Split ratings for the current user
        user_train, user_test = train_test_split(user_ratings, test_size=test_size, random_state=random_state)
        train_data.append(user_train)
        test_data.append(user_test)
    
    # Concatenate the split data back into DataFrames
    train_data = pd.concat(train_data).reset_index(drop=True)
    test_data = pd.concat(test_data).reset_index(drop=True)
    
    # Logging the result of the split
    print(f"Train/Test split completed.")
    print(f"Total users processed: {data['user_id'].nunique()}")
    print(f"Users with fewer than {min_ratings} ratings skipped: {skipped_users}")
    print(f"Training set size: {train_data.shape[0]} ratings")
    print(f"Testing set size: {test_data.shape[0]} ratings")
    
    return train_data, test_data

# Perform train/test split
train_ratings, test_ratings = train_test_split_ratings(final_rating, test_size=0.2, min_ratings=2, random_state=60)

Train/Test split completed.
Total users processed: 890
Users with fewer than 2 ratings skipped: 8
Training set size: 42784 ratings
Testing set size: 11135 ratings


**Train the Model on the Training Data**
* Using train_ratings, create a pivot table and then convert it to a sparse matrix for the KNN model.

In [39]:
# Check the columns to verify if 'title' exists
print(train_ratings.columns)

# Check the first few rows to confirm the structure
print(train_ratings.head())


Index(['user_id', 'ISBN', 'rating', 'title', 'author', 'year', 'publisher',
       'number_of_ratings'],
      dtype='object')
   user_id        ISBN  rating  \
0      254  067976402X       0   
1      254  0439139597       9   
2      254  0345384466       0   
3      254  0345387651       0   
4      254  0375727345       0   

                                              title           author  year  \
0                            Snow Falling on Cedars   David Guterson  1995   
1      Harry Potter and the Goblet of Fire (Book 4)    J. K. Rowling  2000   
2  The Witching Hour (Lives of the Mayfair Witches)        ANNE RICE  1993   
3                             The Cider House Rules      John Irving  1994   
4                             House of Sand and Fog  Andre Dubus III  2000   

           publisher  number_of_ratings  
0  Vintage Books USA                195  
1         Scholastic                126  
2   Ballantine Books                102  
3   Ballantine Books           

In [40]:
def prepare_knn_model(train_data, algorithm='brute', metric='cosine', n_neighbors=5):
    """
    Prepares and trains a KNN model on the training data.
    
    Parameters:
    - train_data (pd.DataFrame): The training ratings data.
    - algorithm (str): The algorithm for NearestNeighbors. Default is 'brute'.
    - metric (str): The distance metric for finding neighbors. Default is 'cosine'.
    - n_neighbors (int): The number of neighbors to consider.
    
    Returns:
    - model (NearestNeighbors): The trained KNN model.
    - train_pivot (pd.DataFrame): The user-item pivot table.
    - train_sparse (csr_matrix): The sparse matrix representation of the pivot table.
    """
    # Step 1: Create pivot table
    train_pivot = train_data.pivot_table(index='title', columns='user_id', values='rating').fillna(0)
    print(f"Pivot table created with shape: {train_pivot.shape}")

    # Step 2: Convert pivot table to sparse matrix
    train_sparse = csr_matrix(train_pivot)
    print(f"Sparse matrix created with shape: {train_sparse.shape}")
    
    # Step 3: Initialize and train the KNN model
    model = NearestNeighbors(algorithm=algorithm, metric=metric, n_neighbors=n_neighbors)
    model.fit(train_sparse)
    print(f"Model trained with {n_neighbors} neighbors, algorithm='{algorithm}', metric='{metric}'")
    
    return model, train_pivot, train_sparse

# Example usage
model, train_pivot, train_sparse = prepare_knn_model(train_ratings, algorithm='brute', metric='cosine', n_neighbors=5)

Pivot table created with shape: (659, 882)
Sparse matrix created with shape: (659, 882)
Model trained with 5 neighbors, algorithm='brute', metric='cosine'


In [42]:
def backtest_recommendations(test_ratings, train_pivot, model, n_recommendations=5):
    """
    Evaluates the KNN recommendation model using test data by calculating hit rate, precision, and recall.
    
    Parameters:
    - test_ratings (pd.DataFrame): The test dataset containing user ratings.
    - train_pivot (pd.DataFrame): The pivot table used to train the model.
    - model (NearestNeighbors): The trained KNN model.
    - n_recommendations (int): Number of recommendations to generate for each book.
    
    Returns:
    - metrics (dict): Dictionary containing hit rate, precision, and recall metrics.
    """
    hits = 0
    total_recommendations = 0
    total_relevant = 0
    
    # Iterate over each user in the test set
    for user_id, user_test_ratings in test_ratings.groupby('user_id'):
        test_books = set(user_test_ratings['title'])  # Books the user actually rated in the test set
        
        # Skip users not in the training data
        if user_id not in train_pivot.columns:
            continue

        # Books the user rated in the training data
        user_train_books = train_pivot[user_id][train_pivot[user_id] > 0].index.tolist()
        
        for book in user_train_books:
            # Check if the book is in the pivot table
            if book in train_pivot.index:
                # Generate recommendations for the book
                distances, suggestions = model.kneighbors(train_pivot.loc[book].values.reshape(1, -1), n_neighbors=n_recommendations + 1)
                recommended_books = [train_pivot.index[suggestions[0][i]] for i in range(1, len(suggestions[0]))]
                
                # Calculate hits, total recommendations, and total relevant books
                hits_for_user = len(set(recommended_books) & test_books)
                hits += hits_for_user
                total_recommendations += n_recommendations
                total_relevant += len(test_books)
    
    # Calculate metrics
    hit_rate = hits / total_recommendations if total_recommendations > 0 else 0
    precision = hits / total_recommendations if total_recommendations > 0 else 0
    recall = hits / total_relevant if total_relevant > 0 else 0

    # Output metrics as a dictionary for easier interpretation
    metrics = {
        'hit_rate': hit_rate,
        'precision': precision,
        'recall': recall
    }
    
    # Logging the results
    print(f"Hit Rate: {metrics['hit_rate']:.4f}")
    print(f"Precision: {metrics['precision']:.4f}")
    print(f"Recall: {metrics['recall']:.4f}")
    
    return metrics

# Example usage
metrics = backtest_recommendations(test_ratings, train_pivot, model, n_recommendations=5)


Hit Rate: 0.0444
Precision: 0.0444
Recall: 0.0103


In [44]:
# Prepare the data for Surprise library
reader = Reader(rating_scale=(0, 10))  # Adjust based on your dataset’s rating scale
data = Dataset.load_from_df(final_rating[['user_id', 'title', 'rating']], reader)

# Initialize the SVD model
svd_model = SVD()

# Perform cross-validation
print("Performing cross-validation with SVD...")
cv_results = cross_validate(svd_model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)

Performing cross-validation with SVD...
Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    3.4237  3.4175  3.4633  3.4549  3.4512  3.4421  0.0181  
MAE (testset)     2.5212  2.5111  2.5445  2.5515  2.5372  2.5331  0.0149  
Fit time          0.68    0.67    0.70    0.67    0.67    0.68    0.01    
Test time         0.10    0.10    0.21    0.10    0.10    0.12    0.04    


In [45]:
# Train SVD on the full dataset
trainset = data.build_full_trainset()
svd_model.fit(trainset)

# Function to get SVD recommendations for a specific user
def get_svd_recommendations(user_id, svd_model, final_rating, n_recommendations=5):
    # List all books
    all_books = final_rating['title'].unique()
    
    # Filter out books the user has already rated
    user_rated_books = final_rating[final_rating['user_id'] == user_id]['title'].unique()
    books_to_predict = [book for book in all_books if book not in user_rated_books]
    
    # Predict ratings for all unrated books
    predictions = [(book, svd_model.predict(user_id, book).est) for book in books_to_predict]
    
    # Sort by estimated rating and return top recommendations
    predictions.sort(key=lambda x: x[1], reverse=True)
    recommended_books = [book for book, _ in predictions[:n_recommendations]]
    
    return recommended_books

# Example usage: Get recommendations for a specific user
user_id = 123  # Replace with an actual user ID
print(f"Top recommendations for user {user_id}:")
print(get_svd_recommendations(user_id, svd_model, final_rating, n_recommendations=5))


Top recommendations for user 123:
["Harry Potter and the Sorcerer's Stone (Book 1)", '84 Charing Cross Road', 'Harry Potter and the Prisoner of Azkaban (Book 3)', 'Harry Potter and the Order of the Phoenix (Book 5)', 'Stupid White Men ...and Other Sorry Excuses for the State of the Nation!']
